In [1]:
import pandas as pd
import kaleido

In [2]:
GEX_SETS_PATH = "/home/mariya/Development/pallas/reports/output/cp/rf_GEX_20260606_190416/FNRCoP/2026-06-24_19-50-20/prediction_sets.csv"
DNAM_SETS_PATH = "/home/mariya/Development/pallas/reports/output/cp/rf_DNAm_20260606_193214/FNRCoP/2026-06-24_21-13-12/prediction_sets.csv"
PALETTE_PATH = "/home/mariya/Development/pallas/configs/class_colors.tsv"
OUTPUT_DIR = "/home/mariya/Development/pallas/reports/output/cp/sankey/"

# Create output directory if it doesn't exist
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Read them in as df
gex_sets = pd.read_csv(GEX_SETS_PATH, index_col=0)
gex_sets.head()

,id,predicted_label,prediction_sets,true_label,prediction_set_softmax_scores,known_class_softmax_scores
dataset,,,,,,
test,SJHBBBM5PY5,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.5266792135775378,0.526679
test,SJHBBBM5TXY,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.6913082574568313,0.691308
test,SJHBBBM67H6,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.4145710304682025,0.414571
test,ALLT-330,"AML, KMT2A-r","AML, KMT2A-r;B-ALL, KMT2A-r","B-ALL, ETV6::RUNX1",0.2012814793797356;0.1696828955378148,0.021721
test,ALLT-339,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, ETV6::RUNX1",0.2792603597484029,0.130439


In [4]:
dnam_sets = pd.read_csv(DNAM_SETS_PATH, index_col=0)
dnam_sets.head()

,id,predicted_label,prediction_sets,true_label,prediction_set_softmax_scores,known_class_softmax_scores
dataset,,,,,,
test,ALL_1381,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.9687306369790928,0.968731
test,ALL_1386,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.8779826119027223,0.877983
test,ALL_1391,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.6950984758360949,0.695098
test,ALL_438,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.928431546374386,0.928432
test,AML_002,"AML, KMT2A-r","AML, KMT2A-r","AML, KMT2A-r",0.6286148927638815,0.628615


In [5]:
# Read in the palette
palette_df = pd.read_csv(PALETTE_PATH, sep="\t", header=None, names=["class", "color"], index_col=0)
palette_df.head()

,color
class,
"AML, KMT2A-r",#2ED9FF
"AML, RUNX1::RUNX1T1",#F8A19F
"AML, CBFB::MYH11",#FEAF16
"AML, NPM1alt",#FA0087
"AML, NUP98-r",#D85FF7


In [6]:
# Read them in as df
gex_sets = pd.read_csv(GEX_SETS_PATH, index_col=0)
dnam_sets = pd.read_csv(DNAM_SETS_PATH, index_col=0)
gex_sets.head()

,id,predicted_label,prediction_sets,true_label,prediction_set_softmax_scores,known_class_softmax_scores
dataset,,,,,,
test,SJHBBBM5PY5,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.5266792135775378,0.526679
test,SJHBBBM5TXY,"B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1","B-ALL, ETV6::RUNX1",0.6913082574568313,0.691308
test,SJHBBBM67H6,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid",0.4145710304682025,0.414571
test,ALLT-330,"AML, KMT2A-r","AML, KMT2A-r;B-ALL, KMT2A-r","B-ALL, ETV6::RUNX1",0.2012814793797356;0.1696828955378148,0.021721
test,ALLT-339,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, ETV6::RUNX1",0.2792603597484029,0.130439


In [9]:
import pandas as pd
import plotly.graph_objects as go
from collections import defaultdict

def build_sankey(df: pd.DataFrame,
                 true_col: str = "true_label",
                 pred_col: str = "prediction_sets",
                 title: str = "Conformal prediction sets"
                 ) -> go.Figure:
    """
    Build a Sankey diagram mapping each true label to all members of its
    prediction_sets (comma-separated string).

    Left nodes  = true labels
    Right nodes = predicted labels (suffixed with ' (pred)' to keep them
                  visually distinct even when the label text is the same)
    """
    # Count flows: (true_label, predicted_label) → count
    flow_counts: dict[tuple[str, str], int] = defaultdict(int)

    for _, row in df.iterrows():
        true = str(row[true_col]).strip()
        preds = [p.strip() for p in str(row[pred_col]).split(";") if p.strip()]
        for pred in preds:
            flow_counts[(true, pred)] += 1

    # Build node lists (left nodes first, then right nodes)
    true_labels = sorted({k[0] for k in flow_counts})
    pred_labels = sorted({k[1] for k in flow_counts})

    # Right-side node labels get a suffix so identical names don't collapse
    pred_labels_display = [f"{p} (pred)" for p in pred_labels]

    all_labels = true_labels + pred_labels_display
    label_index = {label: i for i, label in enumerate(all_labels)}

    sources, targets, values = [], [], []
    for (true, pred), count in flow_counts.items():
        sources.append(label_index[true])
        targets.append(label_index[f"{pred} (pred)"])
        values.append(count)

    # generate palette from the palette_df
    palette = palette_df["color"].tolist()
    node_colors = [palette[i % len(palette)] for i in range(len(all_labels))]

    # Extend palette so all "(pred)" nodes have the same color as their corresponding true label node
    for i, label in enumerate(all_labels):
        if label.endswith(" (pred)"):
            true_label = label[:-7]  # remove " (pred)" suffix
            if true_label in true_labels:
                true_index = true_labels.index(true_label)
                node_colors[i] = node_colors[true_index]

    def hex_to_rgba(hex_color, alpha=0.8):
        hex_color = hex_color.lstrip("#")
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"

    link_colors = [hex_to_rgba(node_colors[s], 0.5) for s in sources]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=15,
            thickness=15,
            line = dict(color = "black", width = 0.5),
            label=all_labels,
            color=node_colors,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors  # color links based on source node
        ),
    ))

    fig.update_layout(
        title_text=title,
        title_font_size=18,
        font_size=13,
        height=500 + 40 * max(len(true_labels), len(pred_labels)),
    )
    return fig


fig = build_sankey(gex_sets, title="Conformal prediction sets for GEX test data")
fig.show()
fig.write_html(f"{OUTPUT_DIR}/sankey_gex.html")
                          # opens in browser

In [10]:
fig = build_sankey(dnam_sets, title="Conformal prediction sets for DNAm test data")
fig.show()
fig.write_html(f"{OUTPUT_DIR}/sankey_dnam.html")